# Finlora Fraud Risk-Scoring: Preprocessing & Feature Engineering

Fourth deliverable in the pipeline: using **only the training set**
(`Data/splits/train.csv`, produced by `notebooks/train_test_split.ipynb`),
engineer new features and fit a preprocessing pipeline. Everything here is
*fit* on train only -- validation and test are never touched -- so the fitted
pipeline can later be applied (`.transform()`, not `.fit_transform()`) to
validation/test without leaking their distribution into training.

**Contents**
1. Load training data
2. Feature engineering
3. Define the feature set
4. Preprocessing pipeline (fit on train only)
5. Save engineered data & fitted pipeline


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

## 1. Load training data

In [2]:
train = pd.read_csv("../Data/splits/train.csv")
print("train shape:", train.shape)
print(f"train fraud rate: {train['is_fraud'].mean():.4%}")
train.head()

train shape: (88200, 28)
train fraud rate: 2.6599%


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,channel,amount,currency,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud,account_holder_name,account_created_date,personal_spend_baseline_usd,has_invalid_amount
0,FLR250302194835,FLR-ACC-101892,Individual,Tier2_Verified,2025-03-02 12:16:49,Sunday,12,MOBILE PURCHASE - CHICKEN REPUBLIC,Chicken Republic,Restaurants,Mobile App,42.97,EUR,1.02,42.00,0,FR,FR,0,DEV-4723570193,0,21,Completed,0,Funke Bello,2025-02-09,87.76,False
1,FLR260108175666,FLR-ACC-100248,Business,Tier2_Verified,2026-01-08 13:49:44,Thursday,13,WEB PURCHASE - LEADWAY ASSURANCE,Leadway Assurance,Insurance,Web Dashboard,162.52,GBP,1.00,162.52,0,GB,GB,0,DEV-4728957461,0,567,Completed,0,Apex Ventures Ltd,2024-06-20,143.09,False
2,FLR250313170522,FLR-ACC-102808,Individual,Tier3_Enhanced,2025-03-13 19:47:22,Thursday,19,WEB PURCHASE - DSTV,DSTV,Subscription/SaaS,Web Dashboard,25168.90,NGN,0.09,279634.98,0,NG,NG,0,NoDevice,-1,364,Completed,0,Chuka Nwosu,2024-03-14,185.76,False
3,FLR250909114349,FLR-ACC-100919,Business,Tier2_Verified,2025-09-09 20:49:51,Tuesday,20,WEB PURCHASE - LEADWAY ASSURANCE,Leadway Assurance,Insurance,Web Dashboard,311.86,GBP,0.20,1551.40,0,GB,GB,0,DEV-4628468714,0,302,Completed,0,Silverline Technologies Co.,2024-11-11,274.75,False
4,FLR250217211644,FLR-ACC-104842,Individual,Tier1_Basic,2025-02-17 06:13:50,Monday,6,SALARY - BLUEWAVE MEDIA,Unknown,Payroll Transfer,Card Not Present,372.62,EUR,9.50,39.23,0,FR,FR,0,NoDevice,-1,953,Completed,0,Femi Williams,2022-07-10,44.49,False


## 2. Feature engineering

New features derived from columns the EDA notebook already looked at, built
**only from `train`** -- no statistic here is computed from validation or test:

- `hour_bucket`: `hour_of_day` collapsed into Night / Morning / Afternoon / Evening. A coarser, ordinal-free view of time-of-day for the categorical encoder to pick up, alongside the raw hour.
- `is_weekend`: 1 for Saturday/Sunday, 0 otherwise -- `day_of_week` itself showed almost no fraud-rate spread in the EDA, but weekday-vs-weekend is a coarser, more common split worth checking in its own right.
- `amount_log`: `log1p(amount)` (negative amounts, already flagged as `has_invalid_amount`, are clipped to 0 first). `amount` is heavily right-skewed (the EDA box plots needed a log axis to be readable at all); this gives models that don't do their own log-scaling (e.g. Logistic Regression) a compressed version of the same signal.
- `velocity_flag`: `(transaction_velocity_1h >= 1)` as an explicit 0/1 flag. The EDA found `transaction_velocity_1h` is the strongest numeric signal in the dataset almost entirely because of this 0-vs-nonzero split (38.22% of fraud vs. 0.09% of legitimate transactions have a repeat transaction in the prior hour) -- making that split its own feature lets a linear model use it directly instead of relying on a single raw count threshold.

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    hour_bins = [-1, 5, 11, 17, 23]
    hour_labels = ["Night", "Morning", "Afternoon", "Evening"]
    df["hour_bucket"] = pd.cut(df["hour_of_day"], bins=hour_bins, labels=hour_labels).astype(str)

    df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))

    df["velocity_flag"] = (df["transaction_velocity_1h"] >= 1).astype(int)

    return df


train_fe = engineer_features(train)
train_fe[["hour_of_day", "hour_bucket", "day_of_week", "is_weekend", "amount", "amount_log",
          "transaction_velocity_1h", "velocity_flag"]].head()

,hour_of_day,hour_bucket,day_of_week,is_weekend,amount,amount_log,transaction_velocity_1h,velocity_flag
0,12,Afternoon,Sunday,1,42.97,3.783508,0,0
1,13,Afternoon,Thursday,0,162.52,5.096935,0,0
2,19,Evening,Thursday,0,25168.90,10.133404,0,0
3,20,Evening,Tuesday,0,311.86,5.745756,0,0
4,6,Morning,Monday,0,372.62,5.923239,0,0


In [4]:
print("New engineered columns, quick sanity check:")
print(train_fe["hour_bucket"].value_counts())
print()
print(train_fe["is_weekend"].value_counts())
print()
print(train_fe["velocity_flag"].value_counts())
print()
print(train_fe["amount_log"].describe())

New engineered columns, quick sanity check:
hour_bucket
Afternoon    30331
Morning      28068
Evening      25224
Night         4577
Name: count, dtype: int64

is_weekend
0    62526
1    25674
Name: count, dtype: int64

velocity_flag
0    87230
1      970
Name: count, dtype: int64

count    88200.000000
mean         8.268876
std          4.331281
min          0.000000
25%          4.169877
50%          8.696176
75%         11.703596
max         19.410822
Name: amount_log, dtype: float64


## 3. Define the feature set

Same leakage-safe exclusions established in the EDA notebook: `status` is
dropped (a post-hoc leakage risk, not a genuine predictive feature), and
identifier/free-text columns (`transaction_id`, `account_id`,
`account_holder_name`, `description`, `merchant_name`, `device_id`,
`timestamp`, `currency`, `account_created_date`) are dropped -- they either
uniquely identify a row or duplicate a column already kept
(`currency` duplicates `home_country`; `hour_of_day`/`day_of_week` are kept
alongside their engineered bucket/weekend versions since the raw and
engineered forms capture slightly different things).

In [5]:
categorical_features = [
    "account_type", "kyc_tier", "channel", "merchant_category",
    "transaction_country", "home_country", "day_of_week", "hour_bucket",
]
numeric_features = [
    "amount", "amount_log", "amount_to_avg_ratio", "avg_transaction_amount_30d",
    "transaction_velocity_1h", "velocity_flag", "account_age_days", "hour_of_day",
    "personal_spend_baseline_usd", "is_cross_border", "is_new_device",
    "is_weekend", "has_invalid_amount",
]
feature_columns = categorical_features + numeric_features
target_column = "is_fraud"

X_train = train_fe[feature_columns]
y_train = train_fe[target_column]

print("Feature matrix:", X_train.shape)
X_train.head()

Feature matrix: (88200, 21)


,account_type,kyc_tier,channel,merchant_category,transaction_country,home_country,day_of_week,hour_bucket,amount,amount_log,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,velocity_flag,account_age_days,hour_of_day,personal_spend_baseline_usd,is_cross_border,is_new_device,is_weekend,has_invalid_amount
0,Individual,Tier2_Verified,Mobile App,Restaurants,FR,FR,Sunday,Afternoon,42.97,3.783508,1.02,42.00,0,0,21,12,87.76,0,0,1,False
1,Business,Tier2_Verified,Web Dashboard,Insurance,GB,GB,Thursday,Afternoon,162.52,5.096935,1.00,162.52,0,0,567,13,143.09,0,0,0,False
2,Individual,Tier3_Enhanced,Web Dashboard,Subscription/SaaS,NG,NG,Thursday,Evening,25168.90,10.133404,0.09,279634.98,0,0,364,19,185.76,0,-1,0,False
3,Business,Tier2_Verified,Web Dashboard,Insurance,GB,GB,Tuesday,Evening,311.86,5.745756,0.20,1551.40,0,0,302,20,274.75,0,0,0,False
4,Individual,Tier1_Basic,Card Not Present,Payroll Transfer,FR,FR,Monday,Morning,372.62,5.923239,9.50,39.23,0,0,953,6,44.49,0,-1,0,False


## 4. Preprocessing pipeline (fit on train only)

`OneHotEncoder` for the categorical features and `StandardScaler` for the
numeric ones, wrapped in a single `ColumnTransformer`. `.fit_transform()` is
called here on `X_train` only -- when validation/test are preprocessed in a
later notebook, this same fitted object is loaded and only `.transform()` is
called on them, so their values never influence the encoder's categories or
the scaler's mean/variance.

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
feature_names_out = preprocessor.get_feature_names_out()

print("Processed train matrix:", X_train_processed.shape)
print("Output feature names (first 15):", list(feature_names_out[:15]))

Processed train matrix: (88200, 60)
Output feature names (first 15): ['cat__account_type_Business', 'cat__account_type_Individual', 'cat__kyc_tier_Tier1_Basic', 'cat__kyc_tier_Tier2_Verified', 'cat__kyc_tier_Tier3_Enhanced', 'cat__channel_API/Integration', 'cat__channel_Card Not Present', 'cat__channel_Card Present', 'cat__channel_Mobile App', 'cat__channel_USSD', 'cat__channel_Web Dashboard', 'cat__merchant_category_ATM Withdrawal', 'cat__merchant_category_Crypto Exchange', 'cat__merchant_category_Electronics', 'cat__merchant_category_Gambling/Gaming']


## 5. Save engineered data & fitted pipeline

Three artifacts, so later notebooks never need to repeat this fitting step:

- `Data/splits/train_features.csv` -- the engineered (pre-encoding) training data, human-readable, for inspection or re-fitting a different pipeline later.
- `artifacts/preprocessor.joblib` -- the **fitted** `ColumnTransformer`, ready to `.transform()` validation/test.
- `artifacts/feature_config.json` -- the exact feature lists and target column name, so downstream notebooks build `X`/`y` the same way.

In [7]:
OUT_DATA_DIR = Path("../Data/splits")
OUT_ARTIFACTS_DIR = Path("../artifacts")
OUT_ARTIFACTS_DIR.mkdir(exist_ok=True)

train_fe.to_csv(OUT_DATA_DIR / "train_features.csv", index=False)
joblib.dump(preprocessor, OUT_ARTIFACTS_DIR / "preprocessor.joblib")

feature_config = {
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "feature_columns": feature_columns,
    "target_column": target_column,
}
with open(OUT_ARTIFACTS_DIR / "feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

print("Saved:")
print(" ", OUT_DATA_DIR / "train_features.csv", f"({train_fe.shape[0]:,} rows, {train_fe.shape[1]} columns)")
print(" ", OUT_ARTIFACTS_DIR / "preprocessor.joblib")
print(" ", OUT_ARTIFACTS_DIR / "feature_config.json")

Saved:
  ..\Data\splits\train_features.csv (88,200 rows, 32 columns)
  ..\artifacts\preprocessor.joblib
  ..\artifacts\feature_config.json
